## Viterbi Algorithm – Nature Primer HMM Setup

In [10]:
# Define model
states = ['E', '5', 'I']
trans = {
    'Start': {'E': 1.0},
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'End': 0.1}
}

emit = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.00, 'G': 0.95, 'T': 0.00},
    'I': {'A': 0.40, 'C': 0.10, 'G': 0.10, 'T': 0.40}
}

In [11]:
import math

def logx(x):
    return -math.inf if x == 0 else math.log(x)

def calc_log_prob(path, sequence):  
    if len(path) != len(sequence):
        raise ValueError("Path and sequence lengths don't match.")
    log_sum = 0.0
    prev = 'Start'
    for i in range(len(sequence)):
        curr = path[i]
        obs = sequence[i]
        trans = trans_mat[prev][curr]
        emit = emit_mat[curr][obs]
        log_sum += logx(trans) + logx(emit)
        prev = curr
    if prev == 'I':
        log_sum += logx(trans_mat[prev]['End'])
    return log_sum

# Input
path_str = "EEEEEEEEEEEEEEEEEE5IIIIIII"
obs_seq = "CTTCATGTGAAAGCAGACGTAAGTCA"

# Output
ans = calc_log_prob(path_str, obs_seq)
print(f"Log probability: {ans}")


Log probability: -41.21967768602254


In [14]:
def viterbi(seq):
    n = len(seq)
    dp = [{}]
    track = {}

    for s in ['E']:
        t = trans['Start'][s]
        e = emit[s][seq[0]]
        dp[0][s] = logx(t) + logx(e)
        track[s] = [s]

    for i in range(1, n):
        dp.append({})
        next_track = {}

        for curr in states:
            max_val = -math.inf
            prev_best = None

            for prev in dp[i - 1]:
                if curr in trans.get(prev, {}):
                    val = dp[i - 1][prev] + logx(trans[prev][curr]) + logx(emit[curr][seq[i]])
                    if val > max_val:
                        max_val = val
                        prev_best = prev

            if prev_best:
                dp[i][curr] = max_val
                next_track[curr] = track[prev_best] + [curr]

        track = next_track

    final_state = max(dp[-1], key=lambda s: dp[-1][s])
    final_prob = round(dp[-1][final_state], 2)
    return ''.join(track[final_state]), final_prob

dna = "CTTCATGTGAAAGCAGACGTAAGTCA"
path, prob = viterbi(dna)

print("Most likely path:", path)
print("Log probability:", prob)

Most likely path: EEEEEEEEEEEEEEEEEEEEEEEEEE
Log probability: -38.68
